# 🦙 Local Text-to-SQL RAG (Ollama & Qwen 2.5)

This notebook demonstrates how to query massive tabular datasets instantly using **DuckDB** and **LangChain**.

We are using **Ollama** to run the AI completely offline on your personal computer! This means **ZERO** rate limits, infinite requests, and completely private data processing.

### Step 1: Install Dependencies

In [1]:
# Install the necessary libraries for our pipeline
#!pip install -q duckdb duckdb-engine langchain langchain-classic langchain-community langchain-ollama sqlalchemy==2.0.44

### Step 2: Build the Database instantly with DuckDB
Instead of loading 1.4M rows into memory (RAM) with Pandas which causes crashes, we build a lightning-fast local analytical database.

In [2]:
import duckdb
import os
import pandas as pd

# File paths
db_path = "apple_sales_rag_ollama.db"
csv_path = "../data/processed/cleaned_apple_sales_enriched_realistic.csv"

# Remove old db if exists to prevent overlapping issues during testing
if os.path.exists(db_path):
    try: os.remove(db_path)
    except: pass

print(f"Connecting to DuckDB and loading massive dataset from {csv_path}...")
con = duckdb.connect(db_path)
con.execute(f"CREATE TABLE sales AS SELECT * FROM read_csv_auto('{csv_path}')")
print("\u2705 Successfully loaded 1 Million rows into DuckDB!")

print("\nSchema (What the Local LLM sees):")
display(con.execute("DESCRIBE sales").df())
con.close()

Connecting to DuckDB and loading massive dataset from ../data/processed/cleaned_apple_sales_enriched_realistic.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Successfully loaded 1 Million rows into DuckDB!

Schema (What the Local LLM sees):


,column_name,column_type,null,key,default,extra
0,sale_id,VARCHAR,YES,None,None,None
1,sale_date,DATE,YES,None,None,None
2,store_id,VARCHAR,YES,None,None,None
3,product_id,VARCHAR,YES,None,None,None
4,quantity,BIGINT,YES,None,None,None
5,product_name,VARCHAR,YES,None,None,None
6,launch_date,DATE,YES,None,None,None
7,price,BIGINT,YES,None,None,None
8,store_name,VARCHAR,YES,None,None,None
9,city,VARCHAR,YES,None,None,None


### Step 3: Advanced AI Prompt Engineering
Because local models don't naturally understand the context of your data, we inject a **Custom System Prompt**. 
This acts as the "brain" or instruction manual for the AI, giving it custom logic hooks for the Apple Retail dataset.

In [3]:
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain_ollama import ChatOllama
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain_core.prompts import PromptTemplate

# 1. Connect LangChain to DuckDB
db = SQLDatabase.from_uri(f"duckdb:///{db_path}")

# 2. Initialize Qwen2.5 
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0)

# 3. MLOPS BEST PRACTICE: The Custom Prompt Template
# This injects strict rules natively into the LangChain Query Engine
custom_prompt = PromptTemplate.from_template(
    """You are an elite DuckDB SQL programming assistant answering questions about Apple Retail Sales data.
    Given an input question, first create a syntactically correct DuckDB query to run, then look at the results of the query and return the answer.
    
    Never query for all the columns from a specific table, only ask for the few relevant columns given the question.
    Be careful to not query for columns that do not exist.
    
    IMPORTANT APPLE RETAIL BUSINESS RULES:
    1. If asked about "Sales", "Revenue", or "Income", ALWAYS default to using the 'sales_amount_realistic' column, NOT the base 'sales_amount' column.
    2. If asked about Volume or Item Counts, ALWAYS use 'quantity_realistic'.
    3. If asked about a Country, ALWAYS filter and select using 'country_norm_mapped'. Never put a country in the 'city' column.
    4. There is ONLY ONE table named 'sales'. DO NOT try to JOIN other tables like 'categories'. All category data is already inside the 'sales' table (e.g. 'category_name').
    5. If asked to count "transactions" or "orders", use COUNT(*), not SUM(quantity).
    6. If comparing metrics across different years side-by-side, use conditional aggregation (e.g. `SUM(CASE WHEN year=2023 THEN sales_amount_realistic END)`).
    7. Avoid markdown wrapping. Output only the raw, executable SQL string.
    8. To find the "most" or "least" of something (like most expensive), NEVER use MAX() or MIN() alongside an unaggregated column. Instead, always use ORDER BY column DESC/ASC LIMIT 1.
    9. When asked for the price of a product, always SELECT 'price_realistic', NEVER 'sales_amount_realistic'.
    10. When filtering for specific product lines (like 'MacBook' or 'iPhone'), filter using product_name LIKE '%MacBook%' instead of category_name = 'MacBook'.
    Only use the following tables:
    {table_info}

    Return a maximum of {top_k} results unless otherwise specified.

    Question: {input}"""
)

# 4. Create the Direct SQL Chain (Text -> SQL -> Result) hooked up to the Prompt
write_query = create_sql_query_chain(llm, db, prompt=custom_prompt)
execute_query = QuerySQLDataBaseTool(db=db)

# 5. Link the writer tool to the executor tool
chain = write_query | execute_query

print("\u2705 Robust Offline SQL Chain (with Advanced Prompting) is ready to roll!")

✅ Robust Offline SQL Chain (with Advanced Prompting) is ready to roll!


d:\anaconda\envs\Apple\Lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(
C:\Users\GM\AppData\Local\Temp\ipykernel_17904\2155630306.py:43: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  execute_query = QuerySQLDataBaseTool(db=db)


### Step 4: The 100% Offline SQL Gauntlet!
Let's test both simple questions and extremely hard Data Science database queries.

In [4]:
def ask_local_ai(question):
    print(f"\nQuestion: {question}")
    print("Thinking...")
    
    sql_query = write_query.invoke({"question": question})
    clean_sql = sql_query.replace("```sql", "").replace("```", "").replace("SQLQuery:", "").strip()
    print(f"\u2699 Generated SQL: {clean_sql}\n")
    
    try:
        result = execute_query.invoke(clean_sql)
        print(f"====== FINAL ANSWER ======\n{result}\n")
    except Exception as e:
        print(f"Error running SQL: {e}\n")

#### Easy Level Tests (Standard Analytics)

In [5]:
ask_local_ai("How many unique stores do we have in our entire dataset?")
ask_local_ai("What are the distinct product categories we sell? List them out.")
ask_local_ai("Which country had the highest number of overall sales transactions (not volume, just number of rows)?")


Question: How many unique stores do we have in our entire dataset?
Thinking...
⚙ Generated SQL: SELECT COUNT(DISTINCT store_id) AS unique_store_count FROM sales;

====== FINAL ANSWER ======
[(75,)]


Question: What are the distinct product categories we sell? List them out.
Thinking...
⚙ Generated SQL: SELECT DISTINCT category_name FROM sales;

====== FINAL ANSWER ======
[('Smartphone',), ('Tablet',), ('Subscription Service',), ('Laptop',), ('Smart Speaker',), ('Wearable',), ('Streaming Device',), ('Audio',), ('Accessories',), ('Desktop',)]


Question: Which country had the highest number of overall sales transactions (not volume, just number of rows)?
Thinking...
⚙ Generated SQL: SELECT country_norm_mapped, COUNT(*) AS transaction_count
FROM sales
GROUP BY country_norm_mapped
ORDER BY transaction_count DESC
LIMIT 5;

====== FINAL ANSWER ======
[('united states', 207728), ('australia', 97280), ('china', 97022), ('japan', 83697), ('canada', 69468)]



#### Medium Level Tests (Mathematical Inference & Data rules)

In [6]:
ask_local_ai("Which country sold the absolute most physical items (volume) out of all the countries combined?")
ask_local_ai("What is the average Apple revenue for the iPhone 14 in Japan in 2024? Remember to use the realistic amount.")



Question: Which country sold the absolute most physical items (volume) out of all the countries combined?
Thinking...
⚙ Generated SQL: SELECT country_norm_mapped, SUM(quantity_realistic) AS total_volume
FROM sales
GROUP BY country_norm_mapped
ORDER BY total_volume DESC
LIMIT 1;

====== FINAL ANSWER ======
[('united states', 1906234)]


Question: What is the average Apple revenue for the iPhone 14 in Japan in 2024? Remember to use the realistic amount.
Thinking...
⚙ Generated SQL: SELECT AVG(sales_amount_realistic) AS average_revenue
FROM sales
WHERE product_name LIKE '%iPhone 14%' AND country_norm_mapped = 'japan' AND year = 2024;

====== FINAL ANSWER ======
[(3786.4497816593885,)]



#### Advanced Level Tests (HAVING clauses & Time Series Grouping)

In [7]:
ask_local_ai("Show me the top 3 stores with the highest average promo_flag impact, but filter out any store with less than 1000 total sales transactions.")
ask_local_ai("What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.")


Question: Show me the top 3 stores with the highest average promo_flag impact, but filter out any store with less than 1000 total sales transactions.
Thinking...
⚙ Generated SQL: SELECT store_name, AVG(promo_flag) AS avg_promo_impact
FROM sales
GROUP BY store_name
HAVING COUNT(*) >= 1000
ORDER BY avg_promo_impact DESC
LIMIT 3;

====== FINAL ANSWER ======
[('Apple Kumamoto', 0.10470605063653268), ('Apple Piazza Liberty', 0.10467248587985037), ('Apple Opera', 0.10458596894767108)]


Question: What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.
Thinking...
⚙ Generated SQL: SELECT 
    month,
    SUM(sales_amount_realistic) AS total_revenue
FROM 
    sales
WHERE 
    product_name LIKE '%MacBook%' AND country_norm_mapped = 'united states' AND year = 2024
GROUP BY 
    month
ORDER BY 
    month;

====== FINAL ANSWER ======
[(1, 3527250.1325999983), (2, 4347768.806099995), (3, 4631489.109899996), (4, 4277351.9151)

In [8]:
ask_local_ai("top 5 stores")


Question: top 5 stores
Thinking...
⚙ Generated SQL: SELECT store_name, COUNT(*) AS transaction_count
FROM sales
GROUP BY store_name
ORDER BY transaction_count DESC
LIMIT 5;

====== FINAL ANSWER ======
[('Apple Chadstone', 27851), ('Apple Covent Garden', 27803), ('Apple The Dubai Mall', 27742), ('Apple Central World', 27576), ('Apple Orchard Road', 27487)]



In [9]:
ask_local_ai("top 10 products")


Question: top 10 products
Thinking...
⚙ Generated SQL: SELECT product_name, SUM(sales_amount_realistic) AS total_sales
FROM sales
GROUP BY product_name
ORDER BY total_sales DESC
LIMIT 10;

====== FINAL ANSWER ======
[('Mac Pro (2023)', 607205237.0010003), ('Mac Pro (Tower)', 401396329.46000063), ('iMac Pro', 336125021.5480004), ('Mac Pro (Rack)', 264433716.6300004), ('MacBook Pro 16-inch', 178441292.42100012), ('MacBook Pro 14-inch', 161289045.93459982), ('Mac Studio', 113944363.31799988), ('iMac with Retina Display', 101917870.72579983), ('MacBook', 94518015.96299993), ('MacBook (Retina)', 91208806.9679999)]



In [10]:
ask_local_ai("top city")


Question: top city
Thinking...
⚙ Generated SQL: SELECT city FROM sales GROUP BY city ORDER BY SUM(quantity_realistic) DESC LIMIT 5;

====== FINAL ANSWER ======
[('London',), ('New York',), ('Paris',), ('Singapore',), ('Dubai',)]



In [11]:
ask_local_ai("how many columns do we have not from sales")


Question: how many columns do we have not from sales
Thinking...
⚙ Generated SQL: SELECT count(*) FROM information_schema.columns WHERE table_name != 'sales'

====== FINAL ANSWER ======
[(0,)]



In [12]:
ask_local_ai("How many total columns are there in the sales table?")


Question: How many total columns are there in the sales table?
Thinking...
⚙ Generated SQL: SELECT count(*) FROM information_schema.columns WHERE table_name = 'sales';

====== FINAL ANSWER ======
[(36,)]



In [13]:
ask_local_ai("how many rows do we have")


Question: how many rows do we have
Thinking...
⚙ Generated SQL: SELECT COUNT(*) FROM sales;

====== FINAL ANSWER ======
[(1040200,)]



In [14]:
ask_local_ai ("what is gdp of london by year")


Question: what is gdp of london by year
Thinking...
⚙ Generated SQL: SELECT country_norm_mapped, SUM(sales_amount_realistic) AS total_revenue, year
FROM sales
WHERE city = 'London'
GROUP BY country_norm_mapped, year;

====== FINAL ANSWER ======
[('united kingdom', 58332568.30000019, 2021), ('united kingdom', 51068263.25966874, 2024), ('united kingdom', 54365484.0, 2020), ('united kingdom', 41101635.05500008, 2022), ('united kingdom', 49334337.523250066, 2023)]



In [15]:
ask_local_ai("i phone 13 and 14 price by year ")


Question: i phone 13 and 14 price by year 
Thinking...
⚙ Generated SQL: SELECT country_norm_mapped, year, AVG(price_realistic) AS average_price
FROM sales
WHERE product_name LIKE '%iPhone%'
GROUP BY country_norm_mapped, year
ORDER BY country_norm_mapped, year;

====== FINAL ANSWER ======
[('australia', 2020, 782.6943407066934), ('australia', 2021, 686.4487819025526), ('australia', 2022, 618.0548068283914), ('australia', 2023, 564.1905752753964), ('australia', 2024, 531.4339545916621), ('austria', 2020, 781.4472573839662), ('austria', 2021, 691.3034343434348), ('austria', 2022, 609.1103030303029), ('austria', 2023, 543.7185682326624), ('austria', 2024, 533.8949308755763), ('canada', 2020, 780.6546457361052), ('canada', 2021, 703.5877076411942), ('canada', 2022, 626.1918570835063), ('canada', 2023, 562.104615384614), ('canada', 2024, 534.2298271604925), ('china', 2020, 763.5330160618679), ('china', 2021, 695.4386464826355), ('china', 2022, 624.608210967935), ('china', 2023, 564.12918966

In [16]:
ask_local_ai("If my budget is $1000, which iPhone models can I afford based on their average realistic price?")


Question: If my budget is $1000, which iPhone models can I afford based on their average realistic price?
Thinking...
⚙ Generated SQL: SELECT product_name, AVG(price_realistic) AS avg_price
FROM sales
WHERE price_realistic <= 1000
GROUP BY product_name
ORDER BY avg_price DESC;

====== FINAL ANSWER ======
[('iMac with Retina Display', 983.4938999999958), ('MacBook', 946.9710000000122), ('MacBook Pro 13-inch', 946.9710000000106), ('iMac 24-inch', 946.9710000000097), ('MacBook (Retina)', 946.971000000009), ('MacBook Pro (Touch Bar)', 917.0537107616839), ('MacBook Air (M2)', 882.0467673387069), ('iPhone 13 Pro', 880.525409055493), ('iPhone 13 Pro Max', 878.1321306679669), ('MacBook Air (M1)', 874.6010312364685), ('iPhone 14 Pro', 863.8106712564544), ('iPhone 14 Pro Max', 856.9205165312256), ('iPhone 12 Pro Max', 855.7625360846787), ('iPad Pro 12.9-inch', 838.2717966547435), ('iPhone 12 Pro', 827.9448545375596), ('MacBook Air (Retina)', 822.6125071783314), ('Apple Watch Hermès', 778.539199

In [17]:
ask_local_ai("I have $3000 to spend. Can I afford to buy both an iPhone 14 and an iPad ?")


Question: I have $3000 to spend. Can I afford to buy both an iPhone 14 and an iPad ?
Thinking...
⚙ Generated SQL: SELECT 
    product_name, 
    sales_amount_realistic
FROM 
    sales
WHERE 
    product_name LIKE '%iPhone%' OR product_name LIKE '%iPad%'
ORDER BY 
    sales_amount_realistic ASC;

====== FINAL ANSWER ======
[('iPad Pro 11-inch', 0.0), ('iPhone SE (3rd Generation)', 0.0), ('iPhone 12 Pro Max', 0.0), ('iPhone 12 Pro Max', 0.0), ('iPad Pro (M2)', 0.0), ('iPhone 14', 0.0), ('iPhone 12', 0.0), ('iPad Pro (M2)', 0.0), ('Leather Case for iPhone', 0.0), ('Smart Cover for iPad', 0.0), ('iPhone 14 Plus', 0.0), ('iPad mini (6th Generation)', 0.0), ('iPhone 12 mini', 0.0), ('iPhone 12', 0.0), ('iPhone 12', 0.0), ('iPhone 14 Pro', 0.0), ('iPhone 14', 0.0), ('iPad mini (5th Generation)', 0.0), ('iPhone 14 Plus', 0.0), ('iPhone 12 Pro Max', 0.0), ('Leather Case for iPhone', 0.0), ('iPad Pro 11-inch', 0.0), ('iPhone 13 Pro', 0.0), ('iPad (10th Generation)', 0.0), ('iPad Pro 11-inch', 0

In [18]:
ask_local_ai("Is it cheaper on average to buy an iPhone 14 in London or in New York?")
ask_local_ai("What was the most expensive item sold at the 'Apple Covent Garden' store?")


Question: Is it cheaper on average to buy an iPhone 14 in London or in New York?
Thinking...
⚙ Generated SQL: SELECT city, AVG(price_realistic) AS avg_price
FROM sales
WHERE product_name LIKE '%iPhone 14%'
GROUP BY city;

====== FINAL ANSWER ======
[('Bangkok', 808.2133620689655), ('Costa Mesa', 806.3701842546063), ('Barcelona', 808.2592592592592), ('San Francisco', 801.2569444444445), ('Glendale', 807.2910321489002), ('Taipei', 810.7549668874173), ('Toronto', 808.7635392829901), ('Kyoto', 793.524959742351), ('Cheltenham', 807.3713850837139), ('Brisbane', 816.817014446228), ('Brooklyn', 803.1009463722397), ('Fukuoka', 804.188679245283), ('Paris', 809.3838383838383), ('Bogota', 804.6497175141243), ('Vienna', 818.9367088607595), ('Amsterdam', 812.7440758293839), ('Philadelphia', 806.7044025157232), ('Macau', 814.8371040723982), ('Milan', 805.2602965403624), ('Ottawa', 811.16), ('Munich', 811.2977346278317), ('Melbourne', 807.0515297906602), ('Dubai', 806.9093799682036), ('Cologne', 811.

In [19]:
ask_local_ai("Are there any MacBooks available that have an average price under $1500 in 2024?")


Question: Are there any MacBooks available that have an average price under $1500 in 2024?
Thinking...
⚙ Generated SQL: SELECT product_name, AVG(price_realistic) AS avg_price
FROM sales
WHERE product_name LIKE '%MacBook%'
  AND year = 2024
GROUP BY product_name
HAVING AVG(price_realistic) < 1500;

====== FINAL ANSWER ======
[('MacBook Air (M1)', 899.0999999999846), ('MacBook Air (Retina)', 655.4438999999967), ('MacBook', 1169.1000000000008), ('MacBook Air (M2)', 786.6639000000116), ('MacBook Pro (Touch Bar)', 1062.1161842615038), ('MacBook Pro 13-inch', 946.9710000000106), ('MacBook (Early 2015)', 1052.1899999999723), ('MacBook (Retina)', 1169.0999999999983)]



In [20]:
ask_local_ai("Are there any iphone available that have an average price under $1500 in 2024?")


Question: Are there any iphone available that have an average price under $1500 in 2024?
Thinking...
⚙ Generated SQL: SELECT product_name, AVG(price_realistic) AS avg_price
FROM sales
WHERE product_name LIKE '%iPhone%' AND year = 2024
GROUP BY product_name
HAVING avg_price < 1500;

====== FINAL ANSWER ======
[('iPhone 14 Pro', 699.0), ('iPhone 14', 499.0), ('iPhone SE (3rd Generation)', 229.0), ('Silicone Case for iPhone', 19.59999999999993), ('iPhone 14 Pro Max', 745.3932107496464), ('iPhone 12 mini', 299.0), ('iPhone 13', 699.0), ('Leather Case for iPhone', 23.59999999999965), ('iPhone 12 Pro Max', 699.0), ('iPhone 13 mini', 399.0), ('iPhone 12 Pro', 730.0443490701001), ('iPhone 13 Pro Max', 935.8446368446369), ('iPhone 13 Pro', 699.0), ('iPhone 14 Plus', 799.0), ('iPhone 12', 429.0)]



In [21]:
ask_local_ai("What is the absolute cheapest product I can buy from the 'Accessories' category?")


Question: What is the absolute cheapest product I can buy from the 'Accessories' category?
Thinking...
⚙ Generated SQL: SELECT product_name, price_realistic
FROM sales
WHERE category_name = 'Accessories'
ORDER BY price_realistic ASC
LIMIT 1;

====== FINAL ANSWER ======
[('Lightning to USB Cable', 15.475618749999997)]



In [22]:
ask_local_ai("Did the average price of the iPhone 13 drop in 2024 compared to 2023?")


Question: Did the average price of the iPhone 13 drop in 2024 compared to 2023?
Thinking...
⚙ Generated SQL: SELECT AVG(price_realistic) AS avg_price_2023, AVG(price_realistic) AS avg_price_2024
FROM sales
WHERE product_name LIKE '%iPhone 13%'
GROUP BY year;

====== FINAL ANSWER ======
[(898.5336787564767, 898.5336787564767), (834.7850808555033, 834.7850808555033), (748.1215452583992, 748.1215452583992), (691.2339205670802, 691.2339205670802), (685.0560657322378, 685.0560657322378)]



In [23]:
ask_local_ai("Which month in 2024 had the cheapest average realistic price for the ipad?")


Question: Which month in 2024 had the cheapest average realistic price for the ipad?
Thinking...
⚙ Generated SQL: SELECT month, AVG(price_realistic) AS avg_price
FROM sales
WHERE year = 2024 AND product_name LIKE '%iPad%'
GROUP BY month
ORDER BY avg_price ASC
LIMIT 1;

====== FINAL ANSWER ======
[(3, 399.1542922423552)]



In [24]:
ask_local_ai("What is the average realistic price of the iPhone 13 and iPhone 14 for each year? Group the results by product and year")


Question: What is the average realistic price of the iPhone 13 and iPhone 14 for each year? Group the results by product and year
Thinking...
⚙ Generated SQL: SELECT 
    country_norm_mapped,
    year,
    AVG(price_realistic) AS avg_price
FROM 
    sales
WHERE 
    product_name LIKE '%iPhone%'
GROUP BY 
    country_norm_mapped, 
    year;

====== FINAL ANSWER ======
[('united arab emirates', 2020, 768.2220878677953), ('colombia', 2020, 795.8177028451001), ('singapore', 2020, 753.1235728676965), ('australia', 2022, 618.0548068283914), ('italy', 2022, 641.1923316062177), ('spain', 2023, 579.7892857142859), ('austria', 2021, 691.3034343434348), ('germany', 2020, 773.6633941093969), ('mexico', 2022, 621.3927583936801), ('united states', 2021, 691.3826136852733), ('china', 2024, 527.4866760168284), ('canada', 2020, 780.6546457361052), ('united states', 2022, 621.4696001106959), ('france', 2023, 559.5633298208629), ('netherlands', 2023, 572.6738396624476), ('united kingdom', 2023, 563.7553